# Manga / Webtoon → Dialogue Text (Colab)

파이프라인: `PDF/이미지 → Koharu RF-DETR → OCR → 언어 판별 → 외국어만 Qwen 번역 → JSONL/TXT`

- 자동 언어 판별 또는 수동 언어 선택 가능
- OCR은 MangaOCR / PaddleOCR / 자동 선택
- 각 단계에서 진단 로그를 출력해 입력 / detector / OCR / 언어판별 / 번역 중 어디서 문제가 생겼는지 확인하기 쉽게 구성


## 1. 패키지 설치 + 저장소 가져오기


In [ ]:
!pip -q install \
    "rfdetr==1.7.0" \
    "safetensors>=0.5" \
    "huggingface_hub>=0.27" \
    "manga-ocr>=0.1.11" \
    "paddlepaddle>=3.0" \
    "paddleocr>=3.0" \
    "transformers>=4.51" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "lingua-language-detector>=2.0" \
    "pymupdf>=1.24" \
    pillow numpy tqdm matplotlib

!rm -rf /content/manga2text_tmp
!git clone -q https://github.com/HisameOgasahara/manga2text_tmp.git /content/manga2text_tmp

print("[Setup] 패키지 설치 및 repo clone 완료")


## 2. 설정

Colab의 **폼(@param)** 에서 값을 바꿀 수 있습니다.

- `AUTO_DETECT_SOURCE_LANGUAGE=True`: 첫 텍스트 영역을 샘플링해 한국어/일본어/중국어/영어를 자동 판단
- `False`: `SOURCE_LANGUAGE` 값을 그대로 사용
- `OCR_BACKEND="auto"`: 일본어는 MangaOCR, 나머지는 PaddleOCR


In [ ]:
import sys
from pathlib import Path

sys.path.append("/content/manga2text_tmp")

from manga2text_pipeline import (
    auto_detect_source_language,
    build_language_detector,
    classify_inputs,
    collect_page_images,
    describe_input_mode,
    load_koharu_detector,
    load_ocr_backend,
    load_translation_model,
    make_preview_images,
    process_pages,
    resolve_ocr_configuration,
    save_results,
)

WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [INPUT_DIR, PAGE_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

AUTO_DETECT_SOURCE_LANGUAGE = True # @param {type:"boolean"}
SOURCE_LANGUAGE = "ko" # @param ["ko", "ja", "zh", "en"]
OCR_BACKEND = "auto" # @param ["auto", "manga", "paddle"]
READING_DIRECTION = "auto" # @param ["auto", "rtl", "ltr"]
PADDLE_DEVICE = "cpu" # @param ["cpu", "gpu"]

ENABLE_TRANSLATION = True # @param {type:"boolean"}
TRANSLATION_MODEL = "Qwen/Qwen3-1.7B" # @param ["Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]
MAX_NEW_TOKENS = 256 # @param {type:"integer"}

INCLUDE_SFX = False # @param {type:"boolean"}
CROP_PADDING = 8 # @param {type:"integer"}
ROW_TOLERANCE = 80 # @param {type:"integer"}
PDF_DPI = 200 # @param {type:"integer"}
PAGE_LIMIT = 0 # @param {type:"integer"}
INPUT_WORKERS = 4 # @param {type:"integer"}
AUTO_LANGUAGE_SAMPLE_CROPS = 3 # @param {type:"integer"}
DEBUG_LOG = True # @param {type:"boolean"}
DEBUG_SAMPLES_PER_PAGE = 3 # @param {type:"integer"}

CLASS_THRESHOLDS = {
    0: 0.25,  # text
    1: 0.20,  # onomatopoeia / SFX
    2: 0.50,  # bubble
    3: 0.50,  # panel
}

if PAGE_LIMIT <= 0:
    PAGE_LIMIT = None

print("[Config]")
print("  auto language :", AUTO_DETECT_SOURCE_LANGUAGE)
print("  manual lang   :", SOURCE_LANGUAGE)
print("  OCR backend   :", OCR_BACKEND)
print("  reading dir   :", READING_DIRECTION)
print("  translation   :", ENABLE_TRANSLATION)
print("  input workers :", INPUT_WORKERS)
print("  debug log     :", DEBUG_LOG)


## 3. 이미지 / PDF 업로드 + 자동 판별 + 썸네일

한 번에 **이미지 1장 / 여러 이미지 / PDF / 이미지+PDF 혼합** 모두 업로드할 수 있습니다. 업로드 직후 입력 형태를 판별하고 썸네일을 보여줍니다.


In [ ]:
import shutil
import matplotlib.pyplot as plt
from google.colab import files

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

for filename, data in uploaded.items():
    (INPUT_DIR / filename).write_bytes(data)

groups = classify_inputs(INPUT_DIR)
input_mode = describe_input_mode(groups)

print("[Input]")
print("  mode        :", input_mode)
print("  images      :", len(groups["images"]))
print("  PDFs        :", len(groups["pdfs"]))
print("  unsupported :", len(groups["unsupported"]))

for path in groups["unsupported"]:
    print("  [WARN] 지원하지 않는 파일:", path.name)

previews = make_preview_images(
    input_dir=INPUT_DIR,
    max_items=8,
    pdf_preview_pages=3,
)

if not previews:
    raise RuntimeError("미리보기 가능한 이미지/PDF가 없습니다.")

columns = min(4, len(previews))
rows = (len(previews) + columns - 1) // columns
plt.figure(figsize=(4 * columns, 5 * rows))

for index, (label, image) in enumerate(previews, start=1):
    plt.subplot(rows, columns, index)
    plt.imshow(image)
    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 4. 페이지 이미지 준비


In [ ]:
import shutil

if PAGE_DIR.exists():
    shutil.rmtree(PAGE_DIR)
PAGE_DIR.mkdir(parents=True, exist_ok=True)

print("[Page preparation]")
print("  workers :", INPUT_WORKERS)
print("  PDF DPI :", PDF_DPI)
print("  limit   :", PAGE_LIMIT)

page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
    workers=INPUT_WORKERS,
)

print("  prepared pages:", len(page_paths))
for path in page_paths[:10]:
    print("   -", path)

if not page_paths:
    raise RuntimeError("처리할 페이지가 없습니다.")


## 5. Koharu RF-DETR 다운로드 + 로드

여기서 **텍스트 영역 검출 모델**을 로드합니다. 셀 출력에 가중치 경로와 CUDA 사용 가능 여부가 표시됩니다.


In [ ]:
import torch

print("[Detector load]")
detector = load_koharu_detector()

print("[Detector status]")
print("  model : Koharu Layout RF-DETR Seg 2XL")
print("  CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  GPU   :", torch.cuda.get_device_name(0))


## 6. 원문 언어 결정 + OCR 경로 결정

자동 판별을 켜면 첫 몇 개 검출 영역에 대해 여러 OCR 후보를 시험합니다. **어떤 OCR이 어떤 문자열을 읽었고 점수가 얼마인지 출력**되므로 자동 판별이 틀렸을 때 바로 확인할 수 있습니다.


In [ ]:
if AUTO_DETECT_SOURCE_LANGUAGE:
    selected_source_language, auto_language_details = auto_detect_source_language(
        page_paths=page_paths,
        detector=detector,
        class_thresholds=CLASS_THRESHOLDS,
        crop_padding=CROP_PADDING,
        max_crops=AUTO_LANGUAGE_SAMPLE_CROPS,
        paddle_device=PADDLE_DEVICE,
    )
else:
    selected_source_language = SOURCE_LANGUAGE
    auto_language_details = None
    print("[Language] manual selection:", selected_source_language)

ocr_config = resolve_ocr_configuration(
    source_language=selected_source_language,
    ocr_backend=OCR_BACKEND,
    reading_direction=READING_DIRECTION,
)

resolved_ocr_backend = ocr_config["ocr_backend"]
resolved_paddle_lang = ocr_config["paddle_lang"]
resolved_reading_direction = ocr_config["reading_direction"]

print("\n[Resolved route]")
print("  source language :", selected_source_language)
print("  OCR backend     :", resolved_ocr_backend)
print("  Paddle language :", resolved_paddle_lang)
print("  reading dir     :", resolved_reading_direction)

if resolved_ocr_backend == "manga" and selected_source_language != "ja":
    print("[WARN] MangaOCR은 일본어 전용입니다. OCR_BACKEND='auto' 또는 'paddle'을 권장합니다.")


## 7. OCR 모델 로드


In [ ]:
print("[OCR load]")
ocr_model = load_ocr_backend(
    backend=resolved_ocr_backend,
    paddle_lang=resolved_paddle_lang,
    paddle_device=PADDLE_DEVICE,
)

print("[OCR status]")
print("  backend :", resolved_ocr_backend)
print("  lang    :", resolved_paddle_lang)


## 8. OCR 결과용 언어 판별기 로드


In [ ]:
language_detector, language_to_code = build_language_detector()
print("[Language detector]")
print("  supported: ko / ja / zh / en")
print("  ready    : True")


## 9. 소형 번역 LLM 로드

OCR 결과가 한국어면 번역하지 않고, `ja / zh / en` 등 외국어로 판정된 대사만 한국어로 번역합니다.


In [ ]:
translation_tokenizer = None
translation_model = None

if ENABLE_TRANSLATION:
    translation_tokenizer, translation_model = load_translation_model(
        model_name=TRANSLATION_MODEL,
    )
    print("[Translation status]")
    print("  enabled :", True)
    print("  model   :", TRANSLATION_MODEL)
else:
    print("[Translation status]")
    print("  enabled :", False)


## 10. 전체 파이프라인 실행 + 단계별 진단 로그

각 페이지마다 `검출 영역 수 → OCR 샘플 → 판정 언어 → 번역 여부 → 페이지별 성공 개수`를 출력합니다.


In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=resolved_ocr_backend,
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=resolved_reading_direction,
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
    debug=DEBUG_LOG,
    debug_samples_per_page=DEBUG_SAMPLES_PER_PAGE,
)

print("\n[Done]")
print("  extracted records:", len(records))


## 11. 결과 미리보기


In [ ]:
for record in records[:30]:
    translated = (
        record["korean"]
        if record["korean"] != record["original"]
        else "(번역 생략)"
    )

    print(
        f"[p.{record['page']:03d} / {record['order']:02d}] "
        f"lang={record['language']} "
        f"ocr={record['ocr_backend']}"
    )
    print("  원문:", record["original"])
    print("  번역:", translated)
    print()


## 12. JSONL / TXT 저장 + 다운로드


In [ ]:
jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)

files.download(str(jsonl_path))
files.download(str(txt_path))
